hullas bu loyiha shunday bo'ladi yani rasm yuklangandan keyin uning nimaligini aytib beradi va bu mahsulotlar bir magazinga tegishli bo;ladi hamda bu mahsulotlarning narxi bo'ladi va soni chegaralangan bo'ladi buni python if else lar bilan foydalanib qilaman yani shunday sayt bo'ladi bir hahsulotni nomi topilhgandan keyin yani yuklagandan so'ng topilgandan so'ng u mahsulotni narxi va soni ko'rsatiladi keyin sotib olish mumkin karta raqamlar kiritib input orqali va kirutgandan so'ng sotildi yokida bunday mahsulot yo'q deb chiqadi


# **Kiritilgan rasm asosida uning nomini tasniflab beradigan CNN loyiha yani kiyimlarni nomini chiqarib beradi**

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
import torchvision.transforms as transforms

# Kerakli kutubxonalar o'rnatildi

## Kaggle dan data set yuklab olindi

In [2]:
import kagglehub

path = kagglehub.dataset_download("agrigorev/clothing-dataset-full")

print("Path to dataset files:", path)

100%|██████████| 6.50G/6.50G [01:23<00:00, 83.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/agrigorev/clothing-dataset-full/versions/1


In [3]:
import pandas as pd
import os

csv_path = os.path.join(path, 'images.csv')

# CSV faylini yuklash
df = pd.read_csv(csv_path)
display(df.head())

,image,sender_id,label,kids
0,4285fab0-751a-4b74-8e9b-43af05deee22,124,Not sure,False
1,ea7b6656-3f84-4eb3-9099-23e623fc1018,148,T-Shirt,False
2,00627a3f-0477-401c-95eb-92642cbe078d,94,Not sure,False
3,ea2ffd4d-9b25-4ca8-9dc2-bd27f1cc59fa,43,T-Shirt,False
4,3b86d877-2b9e-4c8b-a6a2-1d87513309d0,189,Shoes,False


In [4]:
df.shape

(5403, 4)

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Agar GPU mavjud bo'lsa (torch.cuda.is_available() True qaytarsa), device o'zgaruvchisi cuda (GPU) ga sozlanadi.
# Agar GPU mavjud bo'lmasa (False qaytarsa), device o'zgaruvchisi cpu (markaziy protsessor) ga sozlanadi.

In [6]:
torch.cuda.is_available()
#  bu buyruq PyTorch kutubxonasi GPU (grafik protsessor) dan foydalana oladimi yoki yo'qligini tekshiradi.
# Agar qurilmangizda GPU bo'lsa va PyTorch uni ishlata olsa, True qaytaradi, aks holda False qaytaradi.

True

## Malumotlarni tayyorlash

In [7]:
df = df[['image', 'label']]
df.head()

,image,label
0,4285fab0-751a-4b74-8e9b-43af05deee22,Not sure
1,ea7b6656-3f84-4eb3-9099-23e623fc1018,T-Shirt
2,00627a3f-0477-401c-95eb-92642cbe078d,Not sure
3,ea2ffd4d-9b25-4ca8-9dc2-bd27f1cc59fa,T-Shirt
4,3b86d877-2b9e-4c8b-a6a2-1d87513309d0,Shoes


In [8]:
print(df['label'].unique())

['Not sure' 'T-Shirt' 'Shoes' 'Shorts' 'Shirt' 'Pants' 'Skirt' 'Other'
 'Top' 'Outwear' 'Dress' 'Body' 'Longsleeve' 'Undershirt' 'Hat' 'Polo'
 'Blouse' 'Hoodie' 'Skip' 'Blazer']


In [9]:
df = df[~df['label'].isin(['Not sure', 'Skip'])]
# Ortiqcha bo'lgan yani kiyim turi nomolum bo'lgan 2 tur qatorlari olib tashlandi
print(f"{df.shape[0]}")

IMG_DIR = os.path.join(path, 'images_compressed')

def check_image(image_name, img_dir):
    path_jpg = os.path.join(img_dir, f'{image_name}.jpg')
    path_png = os.path.join(img_dir, f'{image_name}.png')
    return os.path.exists(path_jpg) or os.path.exists(path_png)

initial_rows = df.shape[0]
df['file_exists'] = df['image'].apply(lambda x: check_image(x, IMG_DIR))
df = df[df['file_exists']].drop(columns=['file_exists'])
print(f"Filterlangandan so'ng df dagi qatorlar soni: {df.shape[0]} qator (o'chirildi {initial_rows - df.shape[0]})")

# Umuman olganda bu funksiyada berilgan manzilda jpg yokida png rasm bor yokida yo'qligi tekshirildi

5163
Filterlangandan so'ng df dagi qatorlar soni: 5163 qator (o'chirildi 0)


In [10]:
data_transforms = transforms.Compose([
    transforms.Resize((224, 224)), # Bu kiritilayotgan rasmlarni bir xil 224 ga 224 qilib kiritadi va bu neyron tarmoqlar uchun standart hisoblanadi
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
# transforms.Compose([...]): Bir nechta tasvir o'zgartirish operatsiyalarini ketma-ket bajarish uchun ularni bitta zanjirga birlashtiradi.
# transforms.Resize((224, 224)): Har bir tasvirni 224x224 piksel o'lchamiga o'zgartiradi.
#  Bu ko'pincha neyron tarmoqlarning kirish qatlami uchun standart o'lcham hisoblanadi.
# transforms.ToTensor(): Tasvirlarni PyTorch Tensor formatiga aylantiradi. Bu format neyron tarmoqlar
#  uchun mos keladi va piksellarni 0-255 diapazonidan 0-1 diapazoniga miqyoslashtiradi.
# transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)): Tasvirning har bir rang kanali (qizil, yashil, ko'k)
#  uchun piksellarning qiymatlarini normalizatsiya qiladi. Bu modelning tezroq va barqarorroq
# o'qishiga yordam beradi. Bunda har bir kanalning o'rtacha qiymati 0.5 ga, standart og'ishi ham 0.5 ga o'rnatiladi.
#  u 0-1 diapazonidagi piksellar qiymatlarini -1 dan 1 gacha diapazonga o'zgartiradi. Har bir rang kanali uchun (0.5 o'rtacha qiymat
# va 0.5 standart og'ish bilan) quyidagi formula qo'llaniladi: (piksel_qiymati - o'rtacha_qiymat) / standart_og'ish. Natijada:
# 0 qiymati -1 ga aylanadi.
# 0.5 qiymati 0 ga aylanadi.
# 1 qiymati 1 ga aylanadi.
# Bu normalizatsiya modelning o'rganish jarayonini barqarorlashtiradi va tezlashtiradi

In [11]:
from sklearn.model_selection import train_test_split

x = df['image']
y = df['label']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

print(f"x_train shape: {x_train.shape}")
print(f"x_test shape: {x_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# random_state=42: Natijaning har doim bir xil bo'lishini ta'minlash uchun tasodifiy urug'ni (seed) belgilaydi.
# stratify=y: y (label) ustunidagi har bir kiyim turining nisbatini o'qitish va test to'plamlarida bir xil saqlashni ta'minlaydi.

x_train shape: (4130,)
x_test shape: (1033,)
y_train shape: (4130,)
y_test shape: (1033,)


In [12]:
from PIL import Image
# Bu PIL (Pillow) kutubxonasidan Image modulini import qiladi, bu tasvirlarni yuklash va qayta ishlash uchun ishlatiladi.

noyob_labels = df['label'].unique() # label ning noyob qiymatlarini saqlab oladi
label_unikal_idx = {label: i for i, label in enumerate(noyob_labels)}
# Har bir noyob kiyim turiga (string formatida) unikal raqamli indeks (0, 1, 2...) tayinlaydigan lug'at
# (label_to_idx) yaratadi. Bu modelga kiyim turlarini raqamlar orqali tushuntirish uchun kerak.

class ClothingDataset(torch.utils.data.Dataset):
    def __init__(self, image_names, labels, img_dir, label_to_idx, transform=None):
        self.image_names = image_names.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.img_dir = img_dir
        self.label_to_idx = label_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        img_name = self.image_names.iloc[idx]

        img_path_jpg = os.path.join(self.img_dir, f'{img_name}.jpg')
        img_path_png = os.path.join(self.img_dir, f'{img_name}.png')

        image = None
        if os.path.exists(img_path_jpg):
            image = Image.open(img_path_jpg).convert('RGB')
        elif os.path.exists(img_path_png):
            image = Image.open(img_path_png).convert('RGB')
        else:
            raise FileNotFoundError(f"Neither JPG nor PNG found for image {img_name} in {self.img_dir}")

        label_str = self.labels.iloc[idx]
        label = self.label_to_idx[label_str]

        if self.transform:
            image = self.transform(image)

        return image, label

train_dataset = ClothingDataset(
    image_names=x_train,
    labels=y_train,
    img_dir=IMG_DIR,
    label_to_idx=label_unikal_idx,
    transform=data_transforms
)

test_dataset = ClothingDataset(
    image_names=x_test,
    labels=y_test,
    img_dir=IMG_DIR,
    label_to_idx=label_unikal_idx,
    transform=data_transforms
)

train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

# Kiyim yorliqlarini raqamlash: Kiyim turlarini (stringlarni) 0, 1, 2... kabi raqamlarga aylantiradi.
# ClothingDataset klassini yaratish: Rasmlarni yuklash, ularni qayta ishlash (hajmini o'zgartirish, ranglarini
# moslash) va raqamlangan yorliqlar bilan birga PyTorch modeliga tayyorlash uchun maxsus 'idish' yaratadi.
# DataLoader obyektlarini yaratish: Bu 'idish'dagi ma'lumotlarni modelga kichik guruhlar (batchlar) shaklida,
# o'qitish va test uchun ajratib beruvchi mexanizmlarni ishga tushiradi.
# O'qitish uchun ma'lumotlar aralashtiriladi, test uchun aralashtirilmaydi.
# Umuman olganda, bu blok PyTorch modelini o'qitishga tayyorlash uchun rasmlar va ularning yorliqlarini
# standart formatga keltirib, samarali yuklashni ta'minlaydi.

In [13]:
misol = next(iter(train_loader))
misol[0].shape

# misol[0].shape: misol o'zgaruvchisi ichida birinchi element (ya'ni misol[0]) tasvirlarning batchini anglatadi.
# .shape esa bu tasvirlar batchining o'lchamini (shaklini) ko'rsatadi.

# torch.Size([64, 3, 224, 224]) natijasi shuni anglatadiki:
# 64: Bu batchdagi tasvirlar soni ( batch_size=64 deb belgilaganmiz).
# 3: Har bir tasvirda 3 ta rang kanali bor (RGB rang modeli: Qizil, Yashil, Moviy).
# 224: Har bir tasvirning balandligi 224 piksel.
# 224: Har bir tasvirning eni 224 piksel.
# Qisqacha qilib aytganda: Bu kod train_loaderdan birinchi 64 ta tasvirni olib, ularning o'lchami
# (64 ta rasm, har biri 3 rangli kanal va 224x224 piksel) qanday ekanligini ko'rsatadi.
# Bu ma'lumotlar modelga qanday formatda kirib kelishini tushunish uchun muhimdir.



torch.Size([64, 3, 224, 224])

In [14]:
class OddiyCNN(nn.Module):  # OddiyCNN klassi oddiy Konvolyutsion Neyron Tarmoq (CNN) modelini aniqlaydi:
    def __init__(self):
        super(OddiyCNN, self).__init__()

        num_output_classes = len(noyob_labels)

        self.net = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),
            # Dropout qatlami. O'qitish paytida neyronlarning 25% ni tasodifiy o'chirib, modelning ortiqcha moslashuvini (overfitting) oldini oladi.
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Flatten, Linear, relu, linear
            nn.Flatten(),
            nn.Linear(32 * 56 * 56, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            # Yana bir Dropout qatlami, bu safar neyronlarning 50% ni o'chiradi.
            nn.Linear(128, num_output_classes)
        )

    def forward(self, x):
      return self.net(x)

  # Umumiy Bu klass rasmlarni kirish qilib oladigan, ulardan konvolyutsiya va pooling yordamida xususiyatlar ajratib oladigan,
  # so'ngra bu xususiyatlar asosida rasmning qaysi kiyim turiga tegishli ekanligini bashorat qiladigan oddiy neyron
  # tarmoq modelini belgilaydi. Dropout qatlamlari modelning umumlashtirish qobiliyatini oshirishga yordam beradi.

  # Bu qatlamlar birgalikda ishlaydi: Conv2d rasmdagi naqshlarni aniqlaydi, MaxPool2d bu naqshlarni soddalashtiradi
  # va Linear qatlamlar soddalashtirilgan naqshlar asosida tasvirning umumiy mazmunini tushunib, tasniflash uchun tayyorlaydi.

### Oldindan o'qitilgan (Pre-trained) ResNet modelini yuklash

Model aniqligini oshirish uchun, biz `torchvision.models` kutubxonasidan oldindan o'qitilgan ResNet modelini yuklaymiz. Bu model ImageNet kabi katta ma'lumotlar to'plamlarida o'qitilgan va tasvirlardan muhim xususiyatlarni ajratib olishga qodir.

In [15]:
import torchvision.models as models

# ResNet18 modelini oldindan o'qitilgan vaznlar bilan yuklash
model_ft = models.resnet18(pretrained=True)

# Oxirgi to'liq bog'langan qatlamni (fully connected layer) o'zgartirish
# num_output_classes global o'zgaruvchidan olinadi
num_output_classes = len(noyob_labels)
num_ftrs = model_ft.fc.in_features
model_ft.fc = nn.Linear(num_ftrs, num_output_classes)

# Modelni device (GPU/CPU) ga o'tkazish
model_ft = model_ft.to(device)

# Loss funksiyasi va optimizer (agar kerak bo'lsa, optimizatorni yangi model parametrlari bilan qayta aniqlash)
criterion_ft = nn.CrossEntropyLoss()
optimizer_ft = optim.Adam(model_ft.parameters(), lr=0.001)

print("ResNet18 modeli muvaffaqiyatli yuklandi va sozladi.")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 171MB/s]


ResNet18 modeli muvaffaqiyatli yuklandi va sozladi.


### Yangi ResNet modeli bilan o'qitish

Endi oldindan o'qitilgan ResNet modelini ma'lumotlarimizda o'qitamiz.

In [16]:
epoxa_ft = 15 # Oldindan o'qitilgan model uchun epoxalar soni

for epox in range(epoxa_ft):
  model_ft.train()
  total_loss_ft = 0
  for data, target in train_loader:
    data, target = data.to(device), target.to(device)

    optimizer_ft.zero_grad()
    output_ft = model_ft(data)
    loss_ft = criterion_ft(output_ft, target)
    loss_ft.backward()
    optimizer_ft.step()

    total_loss_ft += loss_ft.item()

  print(f"ResNet Epoxa: {epox+1}, O'rtacha yo'qotish: {total_loss_ft/len(train_loader)}")

ResNet Epoxa: 1, O'rtacha yo'qotish: 1.033751760996305
ResNet Epoxa: 2, O'rtacha yo'qotish: 0.49144823917975794
ResNet Epoxa: 3, O'rtacha yo'qotish: 0.33104202380547154
ResNet Epoxa: 4, O'rtacha yo'qotish: 0.22714102153594679
ResNet Epoxa: 5, O'rtacha yo'qotish: 0.1556578295162091
ResNet Epoxa: 6, O'rtacha yo'qotish: 0.1565632491157605
ResNet Epoxa: 7, O'rtacha yo'qotish: 0.09539980661983673
ResNet Epoxa: 8, O'rtacha yo'qotish: 0.04786105893838864
ResNet Epoxa: 9, O'rtacha yo'qotish: 0.050999609302156246
ResNet Epoxa: 10, O'rtacha yo'qotish: 0.07504540747747972
ResNet Epoxa: 11, O'rtacha yo'qotish: 0.08564237645612313
ResNet Epoxa: 12, O'rtacha yo'qotish: 0.08276702532401452
ResNet Epoxa: 13, O'rtacha yo'qotish: 0.03597196866758168
ResNet Epoxa: 14, O'rtacha yo'qotish: 0.016295760619239166
ResNet Epoxa: 15, O'rtacha yo'qotish: 0.020120932928358135


### ResNet modelining aniqligini baholash

Oldindan o'qitilgan modelning test ma'lumotlaridagi aniqligini tekshiramiz.

In [17]:
model_ft.eval()

correct_ft = 0
total_ft = 0

with torch.no_grad():
  for data, target in test_loader:
    data, target = data.to(device), target.to(device)
    output_ft = model_ft(data)

    _, predicted_ft = torch.max(output_ft.data, 1)
    total_ft += target.size(0)
    correct_ft += (predicted_ft == target).sum().item()

print(f"ResNet Test aniqligi: {100 * correct_ft / total_ft}%")

ResNet Test aniqligi: 82.4782187802517%


In [18]:
import gradio as gr
import torch.nn.functional as F

# Kiyim turlari ro'yxati (modelning barcha yorliqlari bilan sinxronlashtirildi)
classes = noyob_labels.tolist()

def predict_resnet(img):
    transform_for_prediction = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
    ])
    img = transform_for_prediction(img)
    img = img.unsqueeze(0).to(device) # Batch o'lchamini qo'shish

    model_ft.eval()
    with torch.no_grad():
        output = model_ft(img)
        probabilities = F.softmax(output[0], dim=0)

    return {classes[i]: float(probabilities[i]) for i in range(len(classes))}

interface_resnet = gr.Interface(
    fn=predict_resnet,
    inputs=gr.Image(image_mode='RGB', type="pil"),
    outputs=gr.Label(num_top_classes=len(classes)),
    title="Kiyim Turlarini Tasniflash Modeli (ResNet)",
    description="Kiyim rasmini yuklang, model uni qaysi turga tegishli ekanini aniqlaydi (ResNet asosida)."
)

interface_resnet.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5905d4174f1d50f3be.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [19]:
df['label'].value_counts()

,count
label,
T-Shirt,1011
Longsleeve,699
Pants,692
Shoes,431
Shirt,378
Dress,357
Outwear,312
Shorts,308
Hat,171
